# Preprocessing of application columns for baseline logistic regression
- table - app_train.csv, app_valid.csv
---

Preparing data set for modeling with baseline logistic regression
- removing unnecessary columns
- basic grouping for categorical variables
- imputing missing values and creating missing flags
- creating simple bins for discrete variables if applicable
- maping binary cols to 0, 1 values

## 0. Libarries and data

In [280]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / 'src'

from eda_module import (
    plot_quantitative_distribution, plot_binary_distribution, 
    plot_binary_vs_binary, plot_quantitative_vs_binary, plot_categorical_distribution,
    plot_categorical_vs_binary
)

from preprocess_module import (
    bin_quantitative_var, fit_optbin_var, fit_quantitative_binner, transform_quantitative_binner,
    fit_capper, transform_capper, create_imputed_quantitative_features
)

In [281]:
df_train = pd.read_csv(r"..\..\data\interim\app_train.csv")
print(f"Shape of df_train: {df_train.shape}")
print(f"Unique dtypes of df_train: {df_train.dtypes.unique()}")
df_train.head(10)

Shape of df_train: (215257, 122)
Unique dtypes of df_train: [dtype('int64') dtype('O') dtype('float64')]


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,TARGET
0,285133,Cash loans,F,Y,Y,2,405000.0,1971072.0,68643.0,1800000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,191894,Cash loans,M,N,Y,0,337500.0,508495.5,38146.5,454500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0,0
2,369428,Cash loans,M,N,Y,1,112500.0,110146.5,13068.0,90000.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
3,138717,Cash loans,F,N,Y,2,40500.0,66384.0,3519.0,45000.0,...,0,0,0,0.0,0.0,0.0,1.0,0.0,2.0,0
4,202381,Cash loans,M,Y,N,0,225000.0,298512.0,31801.5,270000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
5,185763,Cash loans,F,N,Y,0,315000.0,417024.0,21964.5,360000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0,0
6,148765,Cash loans,M,Y,Y,0,162000.0,168102.0,11362.5,148500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0
7,448028,Cash loans,F,N,Y,0,135000.0,1129500.0,33025.5,1129500.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
8,428649,Cash loans,F,N,Y,0,157500.0,1260702.0,39712.5,1129500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0
9,345767,Cash loans,F,Y,N,0,180000.0,879480.0,25843.5,630000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0,0


In [282]:
df_valid = pd.read_csv(r"..\..\data\interim\app_valid.csv")
print(f"Shape of df_valid: {df_valid.shape}")
print(f"Unique dtypes of df_valid: {df_valid.dtypes.unique()}")
df_valid.head(10)

Shape of df_valid: (92254, 122)
Unique dtypes of df_valid: [dtype('int64') dtype('O') dtype('float64')]


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,TARGET
0,331034,Cash loans,F,N,Y,0,90000.0,803259.0,23616.0,670500.0,...,0,0,0,0.0,0.0,0.0,1.0,0.0,1.0,0
1,242581,Cash loans,F,N,Y,0,270000.0,1288350.0,41692.5,1125000.0,...,0,0,0,0.0,0.0,0.0,1.0,0.0,4.0,0
2,281583,Cash loans,F,N,Y,0,270000.0,253737.0,25227.0,229500.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
3,255865,Cash loans,F,N,Y,0,144000.0,436032.0,16564.5,360000.0,...,0,0,0,0.0,0.0,1.0,0.0,0.0,2.0,0
4,389379,Revolving loans,M,N,N,2,72000.0,225000.0,11250.0,225000.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
5,455506,Cash loans,M,Y,Y,0,315000.0,521280.0,27423.0,450000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0,0
6,234885,Cash loans,F,Y,Y,1,54000.0,102384.0,6673.5,81000.0,...,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0,0
7,425217,Cash loans,M,Y,N,0,315000.0,545040.0,31288.5,450000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0,0
8,209875,Cash loans,F,N,Y,1,315000.0,254700.0,13567.5,225000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0
9,106521,Cash loans,F,Y,N,0,67500.0,292500.0,13014.0,292500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0


## 1. Removing all unnecessary columns
- technical columns
- columns with one value
- columns with high number of NANs

`SK_ID_CURR` will be removed just before saving.

In [283]:
df_train1 = df_train.copy()
df_valid1 = df_valid.copy()

In [284]:
max_acceptable_nan_share = 0.50

In [285]:
credit_bureau_cols = [
    'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
    'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR'
]

building_cols = [
    'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG',
    'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG',
    'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG',
    'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 

    'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE',
    'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE',
    'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE',
    'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE',

    'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI',
    'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI',
    'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI',
    'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI',

    'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'TOTALAREA_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE'
]

In [286]:
cols_to_keep = ['EXT_SOURCE_1']

cols_to_remove = [
    'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START',

    'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
    'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',

    'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG',
    'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG',
    'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG',
    'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 

    'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE',
    'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE',
    'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE',
    'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE',

    'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI',
    'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI',
    'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI',
    'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI',

    'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'TOTALAREA_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE'
]


In [287]:
df_train1['OWN_CAR_AGE_missing'] = np.where(df_train1['OWN_CAR_AGE'].isna(), 1, 0)
df_valid1['OWN_CAR_AGE_missing'] = np.where(df_valid1['OWN_CAR_AGE'].isna(), 1, 0)

In [288]:
df_train_nans = df_train1.isna().mean().to_frame(name='nan_train').reset_index()
df_valid_nans = df_valid1.isna().mean().to_frame(name='nan_valid').reset_index()

df_nans = df_train_nans.merge(right=df_valid_nans, on='index', how='right')
df_nans[(df_nans['nan_train']>0) | (df_nans['nan_valid']>0)].head(20)

,index,nan_train,nan_valid
8,AMT_ANNUITY,0.000037,0.000043
9,AMT_GOODS_PRICE,0.000869,0.000986
10,NAME_TYPE_SUITE,0.004186,0.004238
20,OWN_CAR_AGE,0.660099,0.659462
27,OCCUPATION_TYPE,0.313486,0.313385
28,CNT_FAM_MEMBERS,0.000005,0.000011
40,EXT_SOURCE_1,0.563852,0.563715
41,EXT_SOURCE_2,0.002156,0.002125
42,EXT_SOURCE_3,0.198275,0.198203
43,APARTMENTS_AVG,0.506725,0.509300


In [289]:
high_nan_cols = list(df_nans[
    (df_nans['nan_train']>max_acceptable_nan_share) 
    | (df_nans['nan_valid']>max_acceptable_nan_share)
]['index'].values)

high_nan_cols

['OWN_CAR_AGE',
 'EXT_SOURCE_1',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BUILD_AVG',
 'COMMONAREA_AVG',
 'ELEVATORS_AVG',
 'ENTRANCES_AVG',
 'FLOORSMIN_AVG',
 'LANDAREA_AVG',
 'LIVINGAPARTMENTS_AVG',
 'LIVINGAREA_AVG',
 'NONLIVINGAPARTMENTS_AVG',
 'NONLIVINGAREA_AVG',
 'APARTMENTS_MODE',
 'BASEMENTAREA_MODE',
 'YEARS_BUILD_MODE',
 'COMMONAREA_MODE',
 'ELEVATORS_MODE',
 'ENTRANCES_MODE',
 'FLOORSMIN_MODE',
 'LANDAREA_MODE',
 'LIVINGAPARTMENTS_MODE',
 'LIVINGAREA_MODE',
 'NONLIVINGAPARTMENTS_MODE',
 'NONLIVINGAREA_MODE',
 'APARTMENTS_MEDI',
 'BASEMENTAREA_MEDI',
 'YEARS_BUILD_MEDI',
 'COMMONAREA_MEDI',
 'ELEVATORS_MEDI',
 'ENTRANCES_MEDI',
 'FLOORSMIN_MEDI',
 'LANDAREA_MEDI',
 'LIVINGAPARTMENTS_MEDI',
 'LIVINGAREA_MEDI',
 'NONLIVINGAPARTMENTS_MEDI',
 'NONLIVINGAREA_MEDI',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE']

In [290]:
cols_to_remove = cols_to_remove + high_nan_cols
cols_to_remove = [col for col in cols_to_remove if col not in cols_to_keep]
cols_to_remove

['WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'AMT_REQ_CREDIT_BUREAU_HOUR',
 'AMT_REQ_CREDIT_BUREAU_DAY',
 'AMT_REQ_CREDIT_BUREAU_WEEK',
 'AMT_REQ_CREDIT_BUREAU_MON',
 'AMT_REQ_CREDIT_BUREAU_QRT',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'YEARS_BUILD_AVG',
 'COMMONAREA_AVG',
 'ELEVATORS_AVG',
 'ENTRANCES_AVG',
 'FLOORSMAX_AVG',
 'FLOORSMIN_AVG',
 'LANDAREA_AVG',
 'LIVINGAPARTMENTS_AVG',
 'LIVINGAREA_AVG',
 'NONLIVINGAPARTMENTS_AVG',
 'NONLIVINGAREA_AVG',
 'APARTMENTS_MODE',
 'BASEMENTAREA_MODE',
 'YEARS_BEGINEXPLUATATION_MODE',
 'YEARS_BUILD_MODE',
 'COMMONAREA_MODE',
 'ELEVATORS_MODE',
 'ENTRANCES_MODE',
 'FLOORSMAX_MODE',
 'FLOORSMIN_MODE',
 'LANDAREA_MODE',
 'LIVINGAPARTMENTS_MODE',
 'LIVINGAREA_MODE',
 'NONLIVINGAPARTMENTS_MODE',
 'NONLIVINGAREA_MODE',
 'APARTMENTS_MEDI',
 'BASEMENTAREA_MEDI',
 'YEARS_BEGINEXPLUATATION_MEDI',
 'YEARS_BUILD_MEDI',
 'COMMONAREA_MEDI',
 'ELEVATORS_MEDI',
 'ENTRANCES_MEDI',
 'FLOORSMAX_MEDI',
 'FLOORSMIN_ME

In [291]:
print(f"df_train1: {df_train1.shape}")
print(f"df_valid1: {df_valid1.shape}")

df_train1: (215257, 123)
df_valid1: (92254, 123)


In [292]:
df_train1 = df_train1.drop(columns=cols_to_remove)
df_valid1 = df_valid1.drop(columns=cols_to_remove)

In [293]:
print(f"df_train1: {df_train1.shape}")
print(f"df_valid1: {df_valid1.shape}")

df_train1: (215257, 68)
df_valid1: (92254, 68)


In [294]:
num_cols = list(df_train1.select_dtypes(include=['Int64', 'float64']).columns)
cat_cols = list(df_train1.select_dtypes(include=['O']).columns)
binary_cols = [col for col in df_train1.columns if df_train1[col].nunique()==2]

print(f"len(num_cols) + len(cat_cols) == df_train1.shape[1]: {(len(num_cols) + len(cat_cols)) == df_train1.shape[1]}")
print(f"Number of binary columns: {len(binary_cols)}")

len(num_cols) + len(cat_cols) == df_train1.shape[1]: True
Number of binary columns: 37


## 2. Basic grouping for categorical columns

Max number of categories withn one variable and min share of minory category:

In [295]:
max_cat_n = 4
min_cat_share = 0.05

In [296]:
df_train2 = df_train1.copy()
df_valid2 = df_valid1.copy()

In [297]:
cat_cols_to_group = [col for col in cat_cols if df_train2[col].nunique() > 4]
print(cat_cols_to_group)

['NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE']


In [298]:
for col in cat_cols_to_group:
    print(f"Variable: {col}\n")
    
    df_train_grouped = df_train2[[col, 'TARGET']].groupby(col).agg(
            n_obs = (col, 'count'),
            target_share = ('TARGET', 'mean')
    )
    df_train_grouped['obs_share'] = round(df_train_grouped['n_obs'] / df_train2.shape[0],4)
    df_train_grouped['target_share'] = df_train_grouped['target_share'].round(4)
    print(df_train_grouped[['n_obs', 'obs_share', 'target_share']])
    print('-' * 60)

Variable: NAME_TYPE_SUITE

                  n_obs  obs_share  target_share
NAME_TYPE_SUITE                                 
Children           2327     0.0108        0.0743
Family            28001     0.1301        0.0734
Group of people     198     0.0009        0.0859
Other_A             614     0.0029        0.0896
Other_B            1248     0.0058        0.0994
Spouse, partner    7879     0.0366        0.0787
Unaccompanied    174089     0.8087        0.0820
------------------------------------------------------------
Variable: NAME_INCOME_TYPE

                       n_obs  obs_share  target_share
NAME_INCOME_TYPE                                     
Businessman                9     0.0000        0.0000
Commercial associate   50106     0.2328        0.0758
Maternity leave            3     0.0000        0.0000
Pensioner              38748     0.1800        0.0539
State servant          15382     0.0715        0.0589
Student                    9     0.0000        0.0000
Unemployed 

In [299]:
categories_to_group = {}
special_grouping_cols = []

for col in cat_cols_to_group:
    print(f"Variable: {col}")
    
    df_train_grouped = df_train2[[col, 'TARGET']].groupby(col).agg(
            n_obs = (col, 'count'),
            target_share = ('TARGET', 'mean')
    )
    df_train_grouped['obs_share'] = round(df_train_grouped['n_obs'] / df_train2.shape[0],4)
    df_train_grouped['target_share'] = df_train_grouped['target_share'].round(4)

    minor_cat_share = df_train_grouped[
        df_train_grouped['obs_share'] < 0.05
    ]['obs_share'].sum()

    categories_to_group[col] = list(df_train_grouped[df_train_grouped['obs_share']< 0.05].index)
    
    if minor_cat_share < 0.05:
        min_acceptable_share = df_train_grouped[df_train_grouped['obs_share']>=0.05]['obs_share'].min()
        other_canidadate = df_train_grouped[df_train_grouped['obs_share']==min_acceptable_share].index[0]
        categories_to_group[col].append(other_canidadate)
        special_grouping_cols.append(col)


    print(f"Share of minor categories: {minor_cat_share}\n")
    print(df_train_grouped[df_train_grouped['obs_share']< 0.05][['n_obs', 'obs_share', 'target_share']])
    print('-' * 60)

Variable: NAME_TYPE_SUITE
Share of minor categories: 0.057

                 n_obs  obs_share  target_share
NAME_TYPE_SUITE                                
Children          2327     0.0108        0.0743
Group of people    198     0.0009        0.0859
Other_A            614     0.0029        0.0896
Other_B           1248     0.0058        0.0994
Spouse, partner   7879     0.0366        0.0787
------------------------------------------------------------
Variable: NAME_INCOME_TYPE
Share of minor categories: 0.0001

                  n_obs  obs_share  target_share
NAME_INCOME_TYPE                                
Businessman           9     0.0000        0.0000
Maternity leave       3     0.0000        0.0000
Student               9     0.0000        0.0000
Unemployed           16     0.0001        0.3125
------------------------------------------------------------
Variable: NAME_EDUCATION_TYPE
Share of minor categories: 0.0465

                     n_obs  obs_share  target_share
NAME_EDUC

In [300]:
print(categories_to_group)
print(len(categories_to_group))

{'NAME_TYPE_SUITE': ['Children', 'Group of people', 'Other_A', 'Other_B', 'Spouse, partner'], 'NAME_INCOME_TYPE': ['Businessman', 'Maternity leave', 'Student', 'Unemployed', 'State servant'], 'NAME_EDUCATION_TYPE': ['Academic degree', 'Incomplete higher', 'Lower secondary', 'Higher education'], 'NAME_FAMILY_STATUS': ['Unknown', 'Widow'], 'NAME_HOUSING_TYPE': ['Co-op apartment', 'Municipal apartment', 'Office apartment', 'Rented apartment', 'With parents'], 'OCCUPATION_TYPE': ['Accountants', 'Cleaning staff', 'Cooking staff', 'HR staff', 'High skill tech staff', 'IT staff', 'Low-skill Laborers', 'Medicine staff', 'Private service staff', 'Realty agents', 'Secretaries', 'Security staff', 'Waiters/barmen staff'], 'ORGANIZATION_TYPE': ['Advertising', 'Agriculture', 'Bank', 'Business Entity Type 1', 'Business Entity Type 2', 'Cleaning', 'Construction', 'Culture', 'Electricity', 'Emergency', 'Government', 'Hotel', 'Housing', 'Industry: type 1', 'Industry: type 10', 'Industry: type 11', 'Indu

In [301]:
print(special_grouping_cols)
print(len(special_grouping_cols))
    

['NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS']
3


### 2.1 Grouping all minor categories into 'other'

In [302]:
df_train2_1 = df_train2.copy()
for col in categories_to_group.keys():
    if col not in special_grouping_cols:
        df_train2_1[col] = np.where(
            df_train2_1[col].isin(categories_to_group[col]),
            'other',
            df_train2_1[col]
        )


In [303]:
for col in cat_cols_to_group:
    print(f"Variable: {col}")
    print(df_train[col].value_counts() / df_train.shape[0])
    print("-"*60)

Variable: NAME_TYPE_SUITE
NAME_TYPE_SUITE
Unaccompanied      0.808750
Family             0.130082
Spouse, partner    0.036603
Children           0.010810
Other_B            0.005798
Other_A            0.002852
Group of people    0.000920
Name: count, dtype: float64
------------------------------------------------------------
Variable: NAME_INCOME_TYPE
NAME_INCOME_TYPE
Working                 0.515588
Commercial associate    0.232773
Pensioner               0.180008
State servant           0.071459
Unemployed              0.000074
Student                 0.000042
Businessman             0.000042
Maternity leave         0.000014
Name: count, dtype: float64
------------------------------------------------------------
Variable: NAME_EDUCATION_TYPE
NAME_EDUCATION_TYPE
Secondary / secondary special    0.710746
Higher education                 0.242752
Incomplete higher                0.033685
Lower secondary                  0.012334
Academic degree                  0.000483
Name: count, dty

In [304]:
for col in cat_cols_to_group:
    print(f"Variable: {col}")
    print(df_train[col].value_counts())
    print("-"*60)

Variable: NAME_TYPE_SUITE
NAME_TYPE_SUITE
Unaccompanied      174089
Family              28001
Spouse, partner      7879
Children             2327
Other_B              1248
Other_A               614
Group of people       198
Name: count, dtype: int64
------------------------------------------------------------
Variable: NAME_INCOME_TYPE
NAME_INCOME_TYPE
Working                 110984
Commercial associate     50106
Pensioner                38748
State servant            15382
Unemployed                  16
Student                      9
Businessman                  9
Maternity leave              3
Name: count, dtype: int64
------------------------------------------------------------
Variable: NAME_EDUCATION_TYPE
NAME_EDUCATION_TYPE
Secondary / secondary special    152993
Higher education                  52254
Incomplete higher                  7251
Lower secondary                    2655
Academic degree                     104
Name: count, dtype: int64
----------------------------------

### 2.2 Handling special grouping strategies

In [305]:
df_train2_2 = df_train2_1.copy()

In [306]:
print(special_grouping_cols)

['NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS']


In [307]:
for col in special_grouping_cols:
    df_train_grouped = df_train2_2[[col, 'TARGET']].groupby(col).agg(
            n_obs = (col, 'count'),
            target_share = ('TARGET', 'mean')
    )
    df_train_grouped['obs_share'] = round(df_train_grouped['n_obs'] / df_train.shape[0],4)
    df_train_grouped['target_share'] = df_train_grouped['target_share'].round(4)

    print(f"Variable: {col}")
    print(df_train_grouped[['n_obs', 'obs_share', 'target_share']])
    print('-'*50, '\n')

Variable: NAME_INCOME_TYPE
                       n_obs  obs_share  target_share
NAME_INCOME_TYPE                                     
Businessman                9     0.0000        0.0000
Commercial associate   50106     0.2328        0.0758
Maternity leave            3     0.0000        0.0000
Pensioner              38748     0.1800        0.0539
State servant          15382     0.0715        0.0589
Student                    9     0.0000        0.0000
Unemployed                16     0.0001        0.3125
Working               110984     0.5156        0.0953
-------------------------------------------------- 

Variable: NAME_EDUCATION_TYPE
                                n_obs  obs_share  target_share
NAME_EDUCATION_TYPE                                           
Academic degree                   104     0.0005        0.0096
Higher education                52254     0.2428        0.0539
Incomplete higher                7251     0.0337        0.0883
Lower secondary                  26

In [308]:
df_train2_2['NAME_INCOME_TYPE'] = np.where(
    df_train2_2['NAME_INCOME_TYPE'].isin(categories_to_group['NAME_INCOME_TYPE'] + ['State servant']),
    'State_servant',
    df_train2_2['NAME_INCOME_TYPE']
)

df_train2_2['NAME_INCOME_TYPE'].value_counts() / df_train2_2.shape[0]

NAME_INCOME_TYPE
Working                 0.515588
Commercial associate    0.232773
Pensioner               0.180008
State_servant           0.071631
Name: count, dtype: float64

In [309]:
df_train2_2['NAME_EDUCATION_TYPE'] = np.where(
    df_train2_2['NAME_EDUCATION_TYPE'].isin(['Academic degree', 'Higher education']),
    'Higher / Academic degree',
    'Secondary / Incomplete higher'
)

df_train2_2['NAME_EDUCATION_TYPE'].value_counts() / df_train2_2.shape[0]

NAME_EDUCATION_TYPE
Secondary / Incomplete higher    0.756765
Higher / Academic degree         0.243235
Name: count, dtype: float64

In [310]:
df_train2_2['edu_higher_academic'] = np.where(
    df_train2_2['NAME_EDUCATION_TYPE'] == 'Higher / Academic degree', 1, 0
)

df_train2_2['edu_secondary_incomp_higher'] = np.where(
    df_train2_2['NAME_EDUCATION_TYPE'] == 'Secondary / Incomplete higher', 1, 0
)

df_train2_2 = df_train2_2.drop(columns=['NAME_EDUCATION_TYPE'])

In [311]:
df_train2_2['NAME_FAMILY_STATUS'] = np.where(
    df_train2_2['NAME_FAMILY_STATUS'].isin(['Civil marriage', 'Single / not married']),
    'Civil marriage / Single / not married',
    np.where(
        df_train2_2['NAME_FAMILY_STATUS'].isin(['Married', 'Unknown']),
        'Married',
        df_train2_2['NAME_FAMILY_STATUS']
    )
)

df_train2_2['NAME_FAMILY_STATUS'].value_counts() / df_train2_2.shape[0]

NAME_FAMILY_STATUS
Married                                  0.638576
Civil marriage / Single / not married    0.244777
Separated                                0.064360
Widow                                    0.052286
Name: count, dtype: float64

In [312]:
df_train2_2

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_YEAR,TARGET,OWN_CAR_AGE_missing,edu_higher_academic,edu_secondary_incomp_higher
0,285133,Cash loans,F,Y,Y,2,405000.0,1971072.0,68643.0,1800000.0,...,0,0,0,0,0,0.0,0,0,1,0
1,191894,Cash loans,M,N,Y,0,337500.0,508495.5,38146.5,454500.0,...,0,0,0,0,0,6.0,0,1,1,0
2,369428,Cash loans,M,N,Y,1,112500.0,110146.5,13068.0,90000.0,...,0,0,0,0,0,NaN,0,1,0,1
3,138717,Cash loans,F,N,Y,2,40500.0,66384.0,3519.0,45000.0,...,0,0,0,0,0,2.0,0,1,0,1
4,202381,Cash loans,M,Y,N,0,225000.0,298512.0,31801.5,270000.0,...,0,0,0,0,0,0.0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215252,297241,Cash loans,F,N,Y,1,157500.0,846517.5,33700.5,684000.0,...,0,0,0,0,0,4.0,0,1,1,0
215253,136325,Revolving loans,F,N,Y,1,135000.0,405000.0,20250.0,405000.0,...,0,0,0,0,0,NaN,0,1,0,1
215254,240509,Cash loans,F,N,N,0,157500.0,272520.0,21528.0,225000.0,...,0,0,0,0,0,4.0,0,1,0,1
215255,387513,Cash loans,F,N,N,0,90000.0,246357.0,24493.5,234000.0,...,0,0,0,0,0,1.0,0,1,0,1
